In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET memory_limit='2GB'")

RAW = "/workspace/data/raw/three_months.csv"
con.sql(f"""
    CREATE OR REPLACE VIEW raw AS
    SELECT * FROM read_csv('{RAW}', header=true, all_varchar=true)
""")

con.sql("SUMMARIZE raw").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,timestamp,VARCHAR,2020-01-10T00:00,2020-31-12T23:59,165147,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
1,http_staus,VARCHAR,200,408,7,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
2,http_host,VARCHAR,mercadolibre.com,mercadolibre.com,1,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
3,http_uri,VARCHAR,/invoices/search?invoice_id=118822000&site_id=...,/invoices/search?invoice_id=229933999&site_id=...,598318,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
4,http_method,VARCHAR,GET,GET,1,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
5,http_referer,VARCHAR,google.com,https://mercadolibre.com/billing,2,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.23
6,http_user_agent,VARCHAR,Mozilla/5.0 (Macintosh; Intel Mac OS XÊx.y; rv...,wget,10,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00
7,source_ip,VARCHAR,1.0.0.1,99.247.33.186,6267,<NA>,<NA>,<NA>,<NA>,<NA>,4478619,0.00


In [2]:
print("== Códigos de estado ==")
display(con.sql("""
    SELECT http_staus, count(*) AS n
    FROM raw GROUP BY 1 ORDER BY n DESC
""").df())

print("== Métodos y hosts ==")
display(con.sql("""
    SELECT http_method, http_host, count(*) AS n
    FROM raw GROUP BY 1, 2 ORDER BY n DESC
""").df())

== Códigos de estado ==


,http_staus,n
0,200,785553
1,400,739695
2,201,739226
3,304,736350
4,401,735324
5,202,733981
6,408,4254
7,406,4236


== Métodos y hosts ==


,http_method,http_host,n
0,GET,mercadolibre.com,4478619


In [3]:
display(con.sql("""
    SELECT
      count(*)                                                        AS filas,
      count(*) FILTER (WHERE try_strptime("timestamp", '%Y-%m-%dT%H:%M') IS NULL) AS ts_invalidos,
      min("timestamp")                                                AS ts_min,
      max("timestamp")                                                AS ts_max,
      count(*) FILTER (WHERE http_uri NOT LIKE '/invoices/search%')   AS otros_endpoints,
      count(*) FILTER (WHERE http_uri NOT LIKE '%authtoken=ATUSER-ID-%') AS sin_token_usuario,
      count(*) FILTER (WHERE http_uri NOT LIKE '%invoice_id=%')       AS sin_invoice_id
    FROM raw
""").df())

display(con.sql("""
    SELECT regexp_extract(http_uri, 'site_id=([^&]+)', 1) AS site_id, count(*) AS n
    FROM raw GROUP BY 1 ORDER BY n DESC
""").df())

,filas,ts_invalidos,ts_min,ts_max,otros_endpoints,sin_token_usuario,sin_invoice_id
0,4478619,2726139,2020-01-10T00:00,2020-31-12T23:59,0,432698,0


,site_id,n
0,MeliCO,1120465
1,MeliMX,1120006
2,MeliBR,1119367
3,MeliAR,1118781


In [4]:
display(con.sql("""
    SELECT
      count(*) AS filas,
      count(*) FILTER (WHERE try_strptime("timestamp", '%Y-%d-%mT%H:%M') IS NULL) AS invalidos_dia_mes,
      count(*) FILTER (WHERE try_strptime("timestamp", '%Y-%m-%dT%H:%M') IS NULL) AS invalidos_mes_dia,
      min(try_strptime("timestamp", '%Y-%d-%mT%H:%M')) AS ts_min,
      max(try_strptime("timestamp", '%Y-%d-%mT%H:%M')) AS ts_max
    FROM raw
""").df())

,filas,invalidos_dia_mes,invalidos_mes_dia,ts_min,ts_max
0,4478619,0,2726139,2020-10-01,2020-12-31 23:59:00


In [5]:
display(con.sql("""
    SELECT
      CASE
        WHEN http_uri NOT LIKE '%authtoken=%' THEN 'sin_authtoken'
        WHEN regexp_extract(http_uri, 'authtoken=([^&]*)', 1) = '' THEN 'authtoken_vacio'
        ELSE 'otro_formato'
      END AS categoria,
      count(*) AS n
    FROM raw
    WHERE http_uri NOT LIKE '%authtoken=ATUSER-ID-%'
    GROUP BY 1 ORDER BY n DESC
""").df())

display(con.sql("""
    SELECT regexp_extract(http_uri, 'authtoken=([^&]*)', 1) AS token, count(*) AS n
    FROM raw
    WHERE http_uri NOT LIKE '%authtoken=ATUSER-ID-%'
    GROUP BY 1 ORDER BY n DESC LIMIT 15
""").df())

display(con.sql("""
    SELECT http_uri FROM raw
    WHERE http_uri NOT LIKE '%authtoken=ATUSER-ID-%'
    LIMIT 5
""").df())

,categoria,n
0,otro_formato,432698


,token,n
0,TEST-ID-websecuser,144425
1,TEST-ID-developuser,144263
2,TEST-ID-websectest,144010


,http_uri
0,/invoices/search?invoice_id=118825851&site_id=...
1,/invoices/search?invoice_id=229933038&site_id=...
2,/invoices/search?invoice_id=118823120&site_id=...
3,/invoices/search?invoice_id=229933503&site_id=...
4,/invoices/search?invoice_id=229933549&site_id=...


In [6]:
display(con.sql("""
    SELECT http_user_agent, count(*) AS n
    FROM raw GROUP BY 1 ORDER BY n DESC
""").df())

display(con.sql("""
    SELECT http_referer, count(*) AS n
    FROM raw GROUP BY 1 ORDER BY n DESC
""").df())

,http_user_agent,n
0,Mozilla/5.0 (Windows NT 10.0) AppleWebKit/537....,883858
1,Mozilla/5.0 (Windows NTÊx.y; Win64; x64; rv:10...,883455
2,Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebK...,883021
3,Mozilla/5.0 (Macintosh; Intel Mac OS XÊx.y; rv...,880728
4,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,880522
5,crawler4j,24140
6,Scrapy/2.3.0 (+https://scrapy.org),24053
7,wget,13768
8,PostmanRuntime/7.25.0,5074


,http_referer,n
0,https://mercadolibre.com/billing,4457578
1,google.com,10634
2,NaN,10407
